# Silver

 notebooku və SQL bağlantısını hazırlayırıq.

In [97]:
import sys
from pathlib import Path
from sqlalchemy import text, Unicode

PROJECT_ROOT = Path.cwd()

while not (PROJECT_ROOT / "src" / "db.py").exists():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise FileNotFoundError("src/db.py tapılmadı")
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))
from src.db import get_engine

engine = get_engine()

with engine.connect() as conn:
    db_name = conn.execute(text("SELECT DB_NAME()")).scalar()

print("Python və SQL Server hazırdır:", db_name)

Python və SQL Server hazırdır: BankAnalyticsDB


## Müştəri və Kredit məlumatı cədvəli

In [98]:
# Bronze-dakı birləşdirilmiş kredit məlumatlarını oxuyuruq
import pandas as pd
with engine.connect() as conn:
    silver_loans_df = pd.read_sql(
        text("SELECT * FROM bronze.loans_enriched"),
        conn
    )

In [99]:
silver_loans_df.head()

,customer_id,age,gender,city,branch,tenure_years,credit_score,balance,salary,products_count,...,interest_rate,status,rejection_reason,defaulted,merge_date,is_holiday,holiday_name,is_weekend,is_workday,is_pre_holiday
0,C00001,39,Qadın,Bakı,Yasamal,2,654,3156.63,908.98,2,...,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN
1,C00002,47,Kişi,Sumqayıt,Sumqayıt Mərkəz,7,779,1327.10,807.53,2,...,13.88,Rədd edildi,Natamam sənədlər,NaN,2026-04-03,0.0,NaN,0.0,1.0,0.0
2,C00002,47,Kişi,Sumqayıt,Sumqayıt Mərkəz,7,779,1327.10,807.53,2,...,6.67,Rədd edildi,Yüksək borc yükü,NaN,2026-05-12,0.0,NaN,0.0,1.0,0.0
3,C00003,18,Qadın,Gəncə,Gəncə Mərkəz,0,675,7973.01,1564.52,3,...,14.47,Təsdiqləndi,NaN,0.0,2026-04-06,0.0,NaN,0.0,1.0,0.0
4,C00003,18,Qadın,Gəncə,Gəncə Mərkəz,0,675,7973.01,1564.52,3,...,14.33,Təsdiqləndi,NaN,0.0,2026-05-22,0.0,NaN,0.0,1.0,0.0


## Datanın təmizlənməsi

In [100]:
silver_loans_df.shape

(6891, 28)

### Sütun tiplərini yoxlayaq

In [101]:
silver_loans_df.dtypes.reset_index()

,index,0
0,customer_id,str
1,age,int64
2,gender,str
3,city,str
4,branch,str
5,tenure_years,int64
6,credit_score,int64
7,balance,float64
8,salary,float64
9,products_count,int64


### application_date ve merge_date sütunlarini date_time tipine keçək

In [102]:
silver_loans_df["application_date"] = pd.to_datetime(
    silver_loans_df["application_date"],
    errors="coerce"
)

silver_loans_df["merge_date"] = pd.to_datetime(
    silver_loans_df["merge_date"],
    errors="coerce"
)

### "age", "credit_score", "products_count", "has_card", "is_active", "churned", "defaulted","tenure_years"sütunlarının tipini İntegere çevirək

In [103]:
sutunlar = [
    "age", "credit_score", "products_count",
    "has_card", "is_active", "churned","defaulted","tenure_years"
]

silver_loans_df[sutunlar] = silver_loans_df[sutunlar].astype("Int64")

In [104]:
pd.set_option("display.max_rows", None)

silver_loans_df.dtypes.to_frame("Data tipi")

,Data tipi
customer_id,str
age,Int64
gender,str
city,str
branch,str
tenure_years,Int64
credit_score,Int64
balance,float64
salary,float64
products_count,Int64


### Məlumatları kiçik hərfə keçirək

In [105]:
string_cols = silver_loans_df.select_dtypes(include=["object", "string"]).columns

silver_loans_df[string_cols] = silver_loans_df[string_cols].apply(
    lambda col: col.str.lower()
)

### Boşluqları silək

In [106]:
from sqlalchemy import Unicode
silver_loans_df.to_sql(
    "loans", engine, schema="silver",
    if_exists="replace", index=False,
    dtype={c: Unicode(255) for c in silver_loans_df.select_dtypes(include=["object", "string"]).columns if c != "merge_date"})

9

### Dublikatları yoxlayaq

In [107]:
silver_loans_df.duplicated().sum()

np.int64(15)

In [108]:
silver_loans_df = silver_loans_df.drop_duplicates()

In [109]:
silver_loans_df.duplicated().sum()

np.int64(0)

### Aşkarlanmış 15 dublikat sətir silindi

## Boş dəyərlərə baxaq

In [110]:
silver_loans_df.isna().sum().reset_index()

,index,0
0,customer_id,0
1,age,0
2,gender,0
3,city,0
4,branch,0
5,tenure_years,0
6,credit_score,0
7,balance,0
8,salary,0
9,products_count,0


### Bir çox sətirdə yaranmış 2076 boş dəyər loans-kredit  cədvəlinin customer-müştəri cədvəlinə birləşməsindən yaranmış boş dəyərlərdir.   

### defaulted-gecikme sütunundakı 3091 boş dəyəri araşdıraq. 2076 sı birləşmədən yaranan boş dəyərlərdir. geriye qalan 1825 boş dəyərin hamısının rədd edilmiş kreditlərə aid olmasını gözləyirik çünki rədd olunmuş kreditlərdə gecikmə olmamalıdır. 

In [111]:
silver_loans_df.groupby("status")["defaulted"].apply(lambda x: x.isna().sum())

status
rədd edildi    1825
təsdiqləndi       0
Name: defaulted, dtype: int64

### 1826 bos deyerin hamisi redd edilen kreditlere aiddir. bu normaldir. 

### Rejection_reason 5142 boş dəyəri araşdıraq.2076 boş dəyər birləşmədən yaranmış boş dəyərlərdir. geriye qalan 3066 boş dəyərlərin  yalnız təsdiqdənmiş kreditlər üçün olmalıdı.təsdiqlənmiş kreditin imtina səbəbi olmur.

In [112]:
silver_loans_df.groupby('status')['rejection_reason'].apply(
    lambda x: x.isna().sum()
).reset_index(name='bos_deyer_sayi')

,status,bos_deyer_sayi
0,rədd edildi,91
1,təsdiqləndi,2975


### 91 rədd olunmuş kreditdə imtina səbəbi qeyd olunmadığı üçün boş götərir. bunları slməyək və "Səbəb qeyd olunmayıb " sözü ilə əvəzləyək

In [113]:
silver_loans_df.loc[
    (silver_loans_df['status'].isin(['Rədd edildi', 'rədd edildi'])) &
    (silver_loans_df['rejection_reason'].isna()),
    'rejection_reason'
] = 'Səbəb qeyd olunmayıb'

In [114]:
silver_loans_df.groupby('status')['rejection_reason'].apply(
    lambda x: x.isna().sum()
).reset_index(name='bos_deyer_sayi')

,status,bos_deyer_sayi
0,rədd edildi,0
1,təsdiqləndi,2975


### geriye qalan bütün boş qalmış imtina xanaları Təsdiqlənən kreditlərə aiddir.

### Interest_rate-Faiz dərəcəsi  bos olan 2126 deyere baxaq

In [115]:
silver_loans_df[silver_loans_df["interest_rate"].isna()]["status"].value_counts(dropna=False)

status
NaN            2076
təsdiqləndi      34
rədd edildi      16
Name: count, dtype: int64

### 2076 boş dəyər birləşmədən yaranan boş dəyərlərdir.16-sı rədd edilib — bunlarda da faizin boş qalması normal haldır. 34-ü boş qalmış faiz dərəcəsi xanası təsdiqlənən kreditlərə aiddir bi isə xoşagələn hal deyil. bu kreditlərin faiz dərəcələrini tapmaq üçün digər kreditləri novlərinə görə qruplaşdıraq faiz dərəcələrinin medianını tapaq və həmin boş qalmış kredit faizlərini novlərinə görə medianlari ilə əvəzləyək

### kredit novlerinin medianlarini tapaq ve bu deyerlere uygun bosluqlari dolduraq

In [116]:
silver_loans_df.groupby("product_type")["interest_rate"].median()

product_type
avtomobil krediti    11.00
biznes krediti       12.13
ipoteka               8.09
istehlak krediti     14.06
Name: interest_rate, dtype: float64

In [117]:
mask = silver_loans_df["status"] == "təsdiqləndi"
medians = silver_loans_df.groupby("product_type")["interest_rate"].transform("median")

silver_loans_df.loc[mask, "interest_rate"] = silver_loans_df.loc[mask, "interest_rate"].fillna(medians)

### yerde qalan bos deyerlerin sayina baxaq

In [118]:
silver_loans_df["interest_rate"].isna().sum()

np.int64(2092)

### Beləliklə yerde qalan boş dəyərlərin hamısı birləşmədən yaranan və rədd olunmuş kreditlərə aiddir.

### Holiday name- Bayram adlarında sütununda boş dəyərlərin yalnız bayram olmayan günlərdə olmasını yoxlayaq

In [119]:
silver_loans_df.groupby("is_holiday")["holiday_name"].apply(lambda x: x.isna().sum())

is_holiday
0.0    4763
1.0       0
Name: holiday_name, dtype: int64

### bayram gunlerinde bos deyer yoxdur

In [120]:
silver_loans_df.shape


(6876, 28)

### Cədvəli sql silver schemasina yazaq

In [121]:
from sqlalchemy import Unicode
silver_loans_df.to_sql(
    "loans", engine, schema="silver",
    if_exists="replace", index=False,
    dtype={c: Unicode(255) for c in silver_loans_df.select_dtypes(include=["object", "string"]).columns if c != "merge_date"}
)

68

In [122]:
silver_loans_df.to_csv(
    "../../data/bank_data/silver_loans.csv",
    index=False,
    encoding="utf-8-sig"
)

# Tranzaksiya və Müştəri məlumatları cədvəli

### SQL Bronze schemasndan transaction enriched faylini oxuyaq

In [123]:
# Bronze-dakı birləşdirilmiş kredit məlumatlarını oxuyuruq
with engine.connect() as conn:
    silver_transactions_df = pd.read_sql(
        text("SELECT * FROM bronze.transactions_enriched"),
        conn
    )
   

In [124]:
silver_transactions_df.head()

,transaction_id,customer_id,date,time,amount,currency,category,channel,age,gender,...,holiday_name,is_weekend,is_workday,is_pre_holiday,rate_currency,rate_base,rate,temperature_2m_mean,precipitation_sum,wind_speed_10m_max
0,T0040264,C02529,3/27/2026,19:55:59,9.91,AZN,Alış-veriş,POS,34,Qadın,...,Əlavə istirahət günü,0,0,0,NaN,NaN,NaN,10.9,35.5,25.1
1,T0040265,C04195,3/27/2026,19:56:53,794.63,AZN,Nağdlaşdırma,ATM,45,Qadın,...,Əlavə istirahət günü,0,0,0,NaN,NaN,NaN,10.9,35.5,25.1
2,T0040266,C03812,3/27/2026,19:57:15,120.90,AZN,Köçürmə,Mobil tətbiq,40,Qadın,...,Əlavə istirahət günü,0,0,0,NaN,NaN,NaN,10.9,35.5,25.1
3,T0040267,C00782,3/27/2026,19:57:17,13.81,AZN,Alış-veriş,POS,48,Qadın,...,Əlavə istirahət günü,0,0,0,NaN,NaN,NaN,10.0,14.3,13.8
4,T0040268,C02866,3/27/2026,19:57:21,36.76,AZN,Alış-veriş,POS,41,Qadın,...,Əlavə istirahət günü,0,0,0,NaN,NaN,NaN,13.1,14.3,11.0


In [125]:
silver_transactions_df.shape

(77396, 32)

### Datanın təmizlənməsi

### Data tiplərinə baxaq

In [126]:
silver_transactions_df.dtypes.to_frame("Data tipi")

,Data tipi
transaction_id,str
customer_id,str
date,str
time,str
amount,float64
currency,str
category,str
channel,str
age,int64
gender,str


### "date" və "merge_date" sütun tiplərini date time formatına çevirək

In [127]:
silver_transactions_df["date"] = pd.to_datetime(
    silver_transactions_df["date"],
    format="%m/%d/%Y"
)

silver_transactions_df["merge_date"] = pd.to_datetime(
    silver_transactions_df["merge_date"]
)

In [128]:
silver_transactions_df[["date", "merge_date"]].dtypes

date          datetime64[us]
merge_date     datetime64[s]
dtype: object

### Məlumatları kiçik hərfə çevirək

In [129]:
string_cols = silver_transactions_df.select_dtypes(
    include=["object", "string"]
).columns

silver_transactions_df[string_cols] = silver_transactions_df[string_cols].apply(
    lambda col: col.map(
        lambda x: x.lower() if isinstance(x, str) else x
    )
)

### Boşluqları silək

In [130]:
silver_transactions_df = silver_transactions_df.apply(
    lambda col: col.map(
        lambda x: x.strip() if isinstance(x, str) else x
    )
)

In [131]:
silver_transactions_df.duplicated().sum()

np.int64(0)

In [132]:
silver_transactions_df.isna().sum().reset_index()

,index,0
0,transaction_id,0
1,customer_id,0
2,date,0
3,time,0
4,amount,0
5,currency,0
6,category,0
7,channel,0
8,age,0
9,gender,0


In [133]:
silver_transactions_df.groupby("currency")["rate"].apply(
    lambda x: x.isna().sum()
)

currency
azn    72100
eur        0
usd        0
Name: rate, dtype: int64

Bu normaldır. Çünki valyuta məzənnəsi cədvəlimizdə USD və EUR üçün məzənnələr var, AZN üçün isə yoxdur.

AZN-in AZN-ə məzənnəsi həmişə 1-dir. Ona görə bu boşluqları 1 ilə doldura bilərik.

In [134]:
mask = silver_transactions_df["currency"] == "AZN"
silver_transactions_df.loc[mask, "rate"] = 1

In [135]:
silver_transactions_df["rate"].isna().sum()

np.int64(72100)

In [136]:
silver_transactions_df.groupby("is_holiday")["holiday_name"].apply(
    lambda x: x.isna().sum())

is_holiday
0    68014
1        0
Name: holiday_name, dtype: int64

bayram gunlerinde bos deyerler yoxdur

In [137]:
silver_transactions_df.isna().sum()[lambda x: x > 0]

holiday_name     68014
rate_currency    72100
rate_base        72100
rate             72100
dtype: int64

In [138]:
mask = silver_transactions_df["currency"] == "AZN"

silver_transactions_df.loc[mask, "rate_currency"] = "AZN"
silver_transactions_df.loc[mask, "rate_base"] = "AZN"

silver_transactions_df.isna().sum()[lambda x: x > 0]

holiday_name     68014
rate_currency    72100
rate_base        72100
rate             72100
dtype: int64

### Cədvəli SQL Silver schemasına yazaq.

In [139]:
silver_transactions_df.to_sql(
    "transactions", engine, schema="silver",
    if_exists="replace", index=False,
    dtype={c: Unicode(255) for c in silver_transactions_df.select_dtypes(include=["object", "string"]).columns if c != "merge_date"}
)

46

In [140]:
silver_transactions_df.to_csv(
    "../../data/bank_data/silver_transactions.csv",
    index=False,
    encoding="utf-8-sig"
)

In [141]:
with engine.connect() as conn:
    silver_tables = pd.read_sql(
        text("""
            SELECT TABLE_NAME
            FROM INFORMATION_SCHEMA.TABLES
            WHERE TABLE_SCHEMA = 'silver'
            ORDER BY TABLE_NAME
        """),
        conn
    )

display(silver_tables)

,TABLE_NAME
0,loans
1,transactions
